In [3]:
import sys
sys.path.append(r"C:\Users\cigp4g\Downloads")  # ✅ Raw string avoids escape issues
import AMC


# Select corners very carfully- switch to 40x zoom before selecting points

In [ ]:
import sys
import pandas as pd
import AMC
import time

# ---------------------------------------
# CONFIGURATION
# ---------------------------------------
AMC_IP = ##########
AXIS_X = 0
AXIS_Y = 1
corner_names = ["Sample Top Left", "Sample Top Right"] # "Sample Bottom Left"] #, "Sample Bottom Left","Sample Bottom Right"]
output_file = "LM_sample_stage_corners.csv"

# ---------------------------------------
# CONNECT TO AMC STAGE
# ---------------------------------------
print(f"Connecting to AMC stage at {AMC_IP} …")
dev = AMC.Device(AMC_IP)
dev.connect()
time.sleep(0.5)
print("Firmware:", dev.system.getFirmwareVersion())

assert dev.status.getStatusConnected(AXIS_X)[1] and dev.status.getStatusConnected(AXIS_Y)[1], \
    "One or both stage axes not connected!"

print("Stage ready. Current pos → X =",
      dev.move.getPosition(AXIS_X)[1],
      "Y =", dev.move.getPosition(AXIS_Y)[1])

# ---------------------------------------
# RECORD STAGE POSITIONS AT SAMPLE CORNERS
# ---------------------------------------
points = []

for name in corner_names:
    input(f"📍 Move the stage to **{name}**, then press Enter to record its position...")
    x = dev.move.getPosition(AXIS_X)[1]
    y = dev.move.getPosition(AXIS_Y)[1]
    print(f"  → {name}: X = {x:.0f} nm, Y = {y:.0f} nm\n")
    points.append({"Name": name, "X": x * 1e-9, "Y": y * 1e-9})  # convert to meters

# ---------------------------------------
# SAVE TO CSV

# ------------------------
#---------------
df = pd.DataFrame(points)
df.to_csv(output_file, index=False)
print(f"✅ Sample corners saved to → {output_file}")
df


Connecting to AMC stage at 192.168.1.1 …
Firmware: 1.3.0
Stage ready. Current pos → X = 2056872 Y = -279104


📍 Move the stage to **Sample Top Left**, then press Enter to record its position... 


  → Sample Top Left: X = 2056869 nm, Y = -279104 nm



📍 Move the stage to **Sample Top Right**, then press Enter to record its position... 


  → Sample Top Right: X = 2143863 nm, Y = -275704 nm

✅ Sample corners saved to → LM_sample_stage_corners.csv


,Name,X,Y
0,Sample Top Left,0.002057,-0.000279
1,Sample Top Right,0.002144,-0.000276


# Image UM_per_Px

In [26]:
import os, time, subprocess
import numpy as np
import cv2
from PIL import Image
import pyautogui

# ── CONFIG ──────────────────────────────────────────────────────────────
OUTPUT_DIR       = r"C:\Image_test\Calibration"
BLANKCAMERA_EXE  = r"C:\Program Files (x86)\Lumenera Corporation\LuCam Capture Software\Executables\BlankCamera.exe"

# GUI click coords for your camera app
CONNECT_BUTTON_COORDS = (874, 412)
CAPTURE_BUTTON_COORDS = (1205, 634)
EXIT_CAPTURE_COORDS   = (1558, 13)
SAVE_BUTTON_COORDS    = (834, 563)
FILENAME_FIELD_COORDS = (700, 450)
EXIT_BUTTON_COORDS    = (1246, 366)

# Crop region (if you want ROI, else None)
CROP_BOX = None   # e.g., (580, 600, 791, 811)

# Known stage shift in microns
SHIFT_UM = 10.0

os.makedirs(OUTPUT_DIR, exist_ok=True)

def snap_frame(tag):
    path = os.path.join(OUTPUT_DIR, f"calib_{tag}.png")
    proc = subprocess.Popen([BLANKCAMERA_EXE])
    try:
        time.sleep(1.2)
        pyautogui.click(*CONNECT_BUTTON_COORDS); time.sleep(0.8)
        pyautogui.click(*CAPTURE_BUTTON_COORDS); time.sleep(1.5)
        pyautogui.click(*EXIT_CAPTURE_COORDS);   time.sleep(1.8)
        pyautogui.click(*SAVE_BUTTON_COORDS);    time.sleep(1.5)
        pyautogui.click(*FILENAME_FIELD_COORDS); time.sleep(0.8)
        pyautogui.hotkey("ctrl","a"); pyautogui.write(path)
        time.sleep(0.2); pyautogui.press("enter"); time.sleep(0.8)
        pyautogui.click(*EXIT_BUTTON_COORDS)
    finally:
        try: proc.terminate()
        except: pass
    return path

def load_gray(path):
    img = Image.open(path).convert("L")
    if CROP_BOX is not None:
        img = img.crop(CROP_BOX)
    return np.array(img, dtype=np.float32)

def compute_shift(img1, img2):
    # Phase correlation → subpixel translation
    shift = cv2.phaseCorrelate(img1 - np.mean(img1), img2 - np.mean(img2))
    dx, dy = shift[0]
    return dx, dy

def main():
    print("Capturing reference image...")
    path1 = snap_frame("before")
    img1 = load_gray(path1)

    input(f"Now move the stage {SHIFT_UM} µm to the RIGHT, then press Enter...")
    path2 = snap_frame("after")
    img2 = load_gray(path2)

    dx, dy = compute_shift(img1, img2)
    pixel_shift = abs(dx)  # use horizontal
    um_per_px = SHIFT_UM / pixel_shift if pixel_shift > 0 else None

    print(f"\nMeasured pixel shift: {pixel_shift:.2f} px (dx={dx:.2f}, dy={dy:.2f})")
    if um_per_px:
        print(f"Calibration: {um_per_px:.4f} µm/px")
    else:
        print("Could not compute calibration (zero shift?)")

if __name__ == "__main__":
    main()


Capturing reference image...


Now move the stage 10.0 µm to the RIGHT, then press Enter... 



Measured pixel shift: 45.94 px (dx=-45.94, dy=-3.11)
Calibration: 0.2177 µm/px
